In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.erp_loc_a101")
display(df)


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-", ""))
display(df)

In [0]:
display(df.select("CNTRY").distinct())

In [0]:
df = df.withColumn(
    "CNTRY", F.when((col("CNTRY") == "USA") | (col("CNTRY") == "US"), "United States")
    .when(col("CNTRY") == "DE","Germany")
    .when((col("CNTRY") == "") | col("CNTRY").isNull(), "Unknown")
    .otherwise(col("CNTRY"))
)


In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "CNTRY": "country",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Writing to Silver Table

In [0]:
df.limit(10).display()

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.erp_customer_location")

In [0]:
%sql
SELECT * FROM baraa_projectwork.silver.erp_customer_location LIMIT 10